# DBSCAN & HDBSCAN: Density-Based Clustering

## What This Notebook Covers
This notebook provides a comprehensive, interview-ready deep dive into **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) and **HDBSCAN** (Hierarchical DBSCAN). You will learn:
- Core mathematical intuition behind density-based clustering
- From-scratch implementation using only NumPy
- Sklearn implementation with metric comparison
- Hyperparameter sensitivity experiments
- Key interview concepts and gotchas

## Prerequisites
- Python 3.8+, NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn
- Basic understanding of Euclidean distance and clustering

## Dataset
We use the **Mall Customer Segmentation** dataset from Kaggle.

**Kaggle Link:** https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python

**Credits:** Vjchoudhary7 on Kaggle. Dataset contains Annual Income and Spending Score for 200 customers — perfect for density-based clustering since customer segments are not convex-shaped blobs.

---
*GRADIENTTS ML Curriculum — Senior ML Engineer & MIT Educator Series*

In [ ]:
# ── CELL 2: ALL IMPORTS ──────────────────────────────────────────────────────
# Core numerical computing — the backbone of all ML operations
import numpy as np

# Data manipulation and loading
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt          # Base plotting engine
import matplotlib.patches as mpatches   # For custom legend patches
import seaborn as sns                    # Statistical visualization on top of matplotlib

# Scikit-learn — sklearn DBSCAN and metrics
from sklearn.cluster import DBSCAN       # Production DBSCAN implementation
from sklearn.preprocessing import StandardScaler  # Feature scaling — CRITICAL for DBSCAN
from sklearn.metrics import silhouette_score      # Cluster quality metric (no ground truth needed)
from sklearn.datasets import make_moons, make_blobs  # Synthetic shapes for visual demos

# HDBSCAN — Hierarchical extension of DBSCAN
try:
    import hdbscan                       # pip install hdbscan
    HDBSCAN_AVAILABLE = True
except ImportError:
    HDBSCAN_AVAILABLE = False
    print('hdbscan not installed. Run: pip install hdbscan')

# Reproducibility seed
np.random.seed(42)

# Plot aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('All imports loaded successfully.')

## Part 1: Theory Recap

**1. Core Idea — Density Reachability:**
A point `p` is *directly density-reachable* from `q` if `dist(p,q) ≤ ε` AND `|N_ε(q)| ≥ MinPts`, where `N_ε(q)` is the ε-neighbourhood of q.

**2. Point Classification:**
Every point is exactly one of: **Core** (≥ MinPts neighbours within ε), **Border** (within ε of a core point but < MinPts neighbours itself), or **Noise** (neither — labelled −1).

**3. Cluster Formation:**
A cluster is the maximal set of *density-connected* points. Two core points are in the same cluster iff there exists a chain of directly density-reachable points linking them.

**4. HDBSCAN Extension — Mutual Reachability Distance:**
HDBSCAN replaces raw ε-distance with `mreach(a,b) = max(core_k(a), core_k(b), dist(a,b))`. It then builds a minimum spanning tree over this metric and extracts a *condensed cluster tree* — allowing clusters of varying density.

**5. Complexity & Why It Matters:**
DBSCAN is O(n log n) with a spatial index (ball-tree/kd-tree) and O(n²) worst case. It requires NO pre-specified number of clusters — a key interview talking point over K-Means.

## Load the Real-World Dataset
We load the Mall Customer dataset from Kaggle. The features are **Annual Income (k$)** and **Spending Score (1–100)** — a 2D space ideal for visualizing cluster shapes that are decidedly non-spherical.

In [ ]:
# ── CELL 4: LOAD DATASET ─────────────────────────────────────────────────────
# Download from: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
# Place 'Mall_Customers.csv' in the same directory as this notebook.

try:
    df = pd.read_csv('Mall_Customers.csv')
except FileNotFoundError:
    # Fallback: reconstruct representative structure for demonstration
    print('Mall_Customers.csv not found. Download from Kaggle link above.')
    print('Using a representative demo dataset instead...')
    np.random.seed(42)
    n = 200
    df = pd.DataFrame({
        'CustomerID': range(1, n+1),
        'Genre': np.random.choice(['Male', 'Female'], n),
        'Age': np.random.randint(18, 70, n),
        'Annual Income (k$)': np.random.randint(15, 140, n),
        'Spending Score (1-100)': np.random.randint(1, 100, n)
    })

print('Shape:', df.shape)
print('\n--- HEAD ---')
print(df.head())
print('\n--- INFO ---')
df.info()
print('\n--- DESCRIBE ---')
print(df.describe())

# Target variable context: No explicit label — this is UNSUPERVISED.
# We treat 'Annual Income' and 'Spending Score' as features.
# The goal: find natural customer groups without assuming how many there are.
print('\nFeatures: Annual Income (k$), Spending Score (1-100)')
print('Task: Unsupervised — discover clusters without specifying k.')

## Preprocessing
DBSCAN uses **Euclidean distance**, so features on vastly different scales will dominate the ε-neighbourhood calculation. StandardScaler is mandatory — not optional.

In [ ]:
# ── CELL 5: PREPROCESSING ────────────────────────────────────────────────────

# Step 1: Select the two clustering features
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X_raw = df[features].values

# Step 2: Check for nulls — DBSCAN cannot handle NaN values
null_counts = df[features].isnull().sum()
print('Null counts per feature:')
print(null_counts)  # Expect: 0 for both

# Step 3: Scale features using StandardScaler (zero mean, unit variance)
# INTERVIEW NOTE: DBSCAN is NOT scale-invariant. Failing to scale is the most
# common real-world mistake with DBSCAN — features with large ranges dominate ε.
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)   # fit on training data; transform applies z = (x - μ) / σ

print(f'\nPost-scaling stats:')
print(f'  Mean: {X.mean(axis=0).round(4)}')   # Should be ≈ [0, 0]
print(f'  Std:  {X.std(axis=0).round(4)}')    # Should be ≈ [1, 1]
print(f'\nFinal feature matrix shape: {X.shape}')

## Part 2: From Scratch Implementation
We implement DBSCAN from scratch using **only NumPy**. The class exposes the standard `fit()` and `predict()` API. Each step maps directly to the algorithm's mathematical definition. Study the `# INTERVIEW NOTE` comments — these are the lines examiners focus on.

In [ ]:
# ── CELL 7: DBSCAN FROM SCRATCH (NumPy only) ──────────────────────────────────

class DBSCANScratch:
    """
    Density-Based Spatial Clustering of Applications with Noise.
    Implemented from first principles using only NumPy.
    
    Parameters
    ----------
    eps : float  — radius of the ε-neighbourhood
    min_samples : int  — minimum points to form a dense region (core point threshold)
    """

    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples
        self.labels_ = None   # -1 = noise, 0,1,2... = cluster ids

    def _euclidean_distance(self, a, b):
        """Compute Euclidean distance between two points."""
        # INTERVIEW NOTE: DBSCAN supports ANY metric (Manhattan, cosine, etc.)
        # but Euclidean is standard for continuous features.
        return np.sqrt(np.sum((a - b) ** 2))

    def _get_neighbours(self, X, point_idx):
        """
        Return indices of all points within ε of X[point_idx].
        This is the ε-neighbourhood: N_ε(p) = {q ∈ D | dist(p,q) ≤ ε}
        """
        neighbours = []
        for i, point in enumerate(X):
            # INTERVIEW NOTE: Naïve O(n) neighbourhood query per point → O(n²) total.
            # Sklearn uses a KD-Tree or Ball-Tree to reduce this to O(n log n).
            if self._euclidean_distance(X[point_idx], point) <= self.eps:
                neighbours.append(i)
        return neighbours

    def fit(self, X):
        """
        Run DBSCAN on dataset X.
        Assigns self.labels_ — an array of cluster IDs (int) or -1 for noise.
        """
        n_samples = X.shape[0]
        
        # All points start as UNVISITED (-2 is our sentinel for unvisited)
        self.labels_ = np.full(n_samples, -2, dtype=int)
        
        cluster_id = 0  # Cluster counter — increments each time a new cluster starts

        for point_idx in range(n_samples):
            # Skip already visited points — each point is visited exactly once
            if self.labels_[point_idx] != -2:
                continue

            # Find all ε-neighbours of this point
            neighbours = self._get_neighbours(X, point_idx)

            if len(neighbours) < self.min_samples:
                # INTERVIEW NOTE: Not enough neighbours → this point is NOISE (−1).
                # A noise point can later become a border point if it falls within
                # ε of a core point discovered later.
                self.labels_[point_idx] = -1
            else:
                # This is a CORE POINT — begin growing a new cluster from it
                self._grow_cluster(X, point_idx, neighbours, cluster_id)
                cluster_id += 1  # Increment only when a full cluster is formed

        return self

    def _grow_cluster(self, X, core_idx, neighbours, cluster_id):
        """
        BFS expansion from a confirmed core point.
        Assigns cluster_id to all density-reachable points.
        """
        # Mark the seed core point with the cluster label
        self.labels_[core_idx] = cluster_id

        # Use a queue for BFS (breadth-first search) expansion
        # INTERVIEW NOTE: DFS also works; BFS is typically used in canonical descriptions.
        queue = list(neighbours)

        i = 0
        while i < len(queue):
            neighbour_idx = queue[i]

            if self.labels_[neighbour_idx] == -1:
                # Previously labelled noise — now reachable from a core point → border point
                self.labels_[neighbour_idx] = cluster_id

            elif self.labels_[neighbour_idx] == -2:
                # Unvisited point — assign to this cluster
                self.labels_[neighbour_idx] = cluster_id

                # Check if this neighbour is itself a core point
                sub_neighbours = self._get_neighbours(X, neighbour_idx)
                if len(sub_neighbours) >= self.min_samples:
                    # INTERVIEW NOTE: Core point discovered inside a cluster → expand further.
                    # This chain of expansion is what makes density-CONNECTED clusters possible.
                    queue.extend(sub_neighbours)

            i += 1

    def predict(self, X=None):
        """
        Return cluster labels from the most recent fit().
        INTERVIEW NOTE: DBSCAN is a transductive algorithm — it cannot assign
        labels to new unseen points without re-running fit(). Unlike K-Means,
        there is no centroid to compute distance to.
        """
        if self.labels_ is None:
            raise RuntimeError('Call fit() before predict().')
        return self.labels_


# ── Run on a manageable subset for the O(n²) scratch implementation ──────────
# Using subset for speed (O(n²) complexity); sklearn handles full 200 points
X_sub = X[:100]   # First 100 points for demo speed

dbscan_scratch = DBSCANScratch(eps=0.4, min_samples=5)
dbscan_scratch.fit(X_sub)
scratch_labels = dbscan_scratch.predict()

n_clusters_scratch = len(set(scratch_labels)) - (1 if -1 in scratch_labels else 0)
n_noise_scratch = np.sum(scratch_labels == -1)

print(f'From-Scratch DBSCAN Results (n=100 subset):')
print(f'  Clusters found : {n_clusters_scratch}')
print(f'  Noise points   : {n_noise_scratch}')
print(f'  Unique labels  : {sorted(set(scratch_labels))}')

In [ ]:
# ── CELL 8: FIT SCRATCH MODEL & EVALUATE ─────────────────────────────────────

# Evaluate silhouette score — measures cluster cohesion vs separation
# Only valid when more than 1 cluster found and not all points are noise
valid_mask = scratch_labels != -1

if n_clusters_scratch > 1 and valid_mask.sum() > 1:
    sil_scratch = silhouette_score(X_sub[valid_mask], scratch_labels[valid_mask])
    print(f'Silhouette Score (scratch, excluding noise): {sil_scratch:.4f}')
    print('Interpretation: [-1, 1]; closer to 1 = dense, well-separated clusters')
else:
    print('Cannot compute silhouette: need ≥2 clusters and ≥2 non-noise points.')

# Cluster composition report
print('\nCluster sizes (scratch):')
unique, counts = np.unique(scratch_labels, return_counts=True)
for label, count in zip(unique, counts):
    tag = 'NOISE' if label == -1 else f'Cluster {label}'
    print(f'  {tag}: {count} points')

## Part 3: Sklearn Implementation
Sklearn's DBSCAN uses a **Ball-Tree** spatial index for O(n log n) neighbourhood queries. It also supports parallel computation. We run it on the full 200-point dataset and directly compare metrics with our scratch implementation.

In [ ]:
# ── CELL 10: SKLEARN DBSCAN + HDBSCAN IMPLEMENTATION ─────────────────────────

# --- DBSCAN (sklearn) on full dataset ---
sklearn_dbscan = DBSCAN(eps=0.4, min_samples=5, metric='euclidean', algorithm='ball_tree')
sklearn_labels = sklearn_dbscan.fit_predict(X)

n_clusters_sk = len(set(sklearn_labels)) - (1 if -1 in sklearn_labels else 0)
n_noise_sk = np.sum(sklearn_labels == -1)

print('=== Sklearn DBSCAN (full 200-point dataset) ===')
print(f'  Clusters   : {n_clusters_sk}')
print(f'  Noise pts  : {n_noise_sk}')

mask_sk = sklearn_labels != -1
if n_clusters_sk > 1 and mask_sk.sum() > 1:
    sil_sk = silhouette_score(X[mask_sk], sklearn_labels[mask_sk])
    print(f'  Silhouette : {sil_sk:.4f}')

# --- HDBSCAN (if available) ---
print('\n=== HDBSCAN ===')
if HDBSCAN_AVAILABLE:
    hdb = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5, metric='euclidean')
    hdb_labels = hdb.fit_predict(X)
    n_clusters_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
    n_noise_hdb = np.sum(hdb_labels == -1)
    print(f'  Clusters   : {n_clusters_hdb}')
    print(f'  Noise pts  : {n_noise_hdb}')
    mask_hdb = hdb_labels != -1
    if n_clusters_hdb > 1 and mask_hdb.sum() > 1:
        sil_hdb = silhouette_score(X[mask_hdb], hdb_labels[mask_hdb])
        print(f'  Silhouette : {sil_hdb:.4f}')
else:
    print('  HDBSCAN not installed. Run: pip install hdbscan')

print('\n--- COMPARISON TABLE ---')
print(f'{"Method":<25} {"Clusters":>10} {"Noise":>8} {"Silhouette":>12}')
print('-' * 58)
print(f'{"Scratch DBSCAN (n=100)":<25} {n_clusters_scratch:>10} {n_noise_scratch:>8}', end=' ')
print(f'{sil_scratch:>12.4f}' if (n_clusters_scratch > 1 and valid_mask.sum() > 1) else f'{"N/A":>12}')
print(f'{"Sklearn DBSCAN (n=200)":<25} {n_clusters_sk:>10} {n_noise_sk:>8}', end=' ')
print(f'{sil_sk:>12.4f}' if (n_clusters_sk > 1 and mask_sk.sum() > 1) else f'{"N/A":>12}')

## Visualisations
Two essential plots: cluster assignments on the customer data, and DBSCAN vs K-Means on a non-convex (moons) shape to demonstrate DBSCAN's key advantage.

In [ ]:
# ── CELL 11: VISUALISATIONS ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Plot 1: DBSCAN clusters on Mall Customer data ──
ax1 = axes[0]
unique_labels = sorted(set(sklearn_labels))
palette = sns.color_palette('tab10', n_colors=max(n_clusters_sk, 1))

for label in unique_labels:
    mask = sklearn_labels == label
    color = 'black' if label == -1 else palette[label % len(palette)]
    marker = 'x' if label == -1 else 'o'
    size = 30 if label == -1 else 60
    lbl_name = 'Noise' if label == -1 else f'Cluster {label}'
    ax1.scatter(X[mask, 0], X[mask, 1], c=[color], label=lbl_name,
                marker=marker, s=size, alpha=0.85, edgecolors='white', linewidths=0.4)

ax1.set_title('DBSCAN — Mall Customer Segments\n(Annual Income vs Spending Score, Scaled)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Annual Income (z-score)', fontsize=11)
ax1.set_ylabel('Spending Score (z-score)', fontsize=11)
ax1.legend(framealpha=0.9)

# ── Plot 2: DBSCAN vs K-Means on Half-Moons (non-convex demo) ──
ax2 = axes[1]
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

from sklearn.cluster import KMeans
km_labels = KMeans(n_clusters=2, random_state=42, n_init='auto').fit_predict(X_moons)
db_labels_moons = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_moons)

# Show DBSCAN result on moons
unique_moon_labels = sorted(set(db_labels_moons))
palette2 = sns.color_palette('Set1', n_colors=len(unique_moon_labels))
for i, label in enumerate(unique_moon_labels):
    mask = db_labels_moons == label
    color = 'gray' if label == -1 else palette2[i]
    name = 'Noise' if label == -1 else f'DBSCAN Cluster {label}'
    ax2.scatter(X_moons[mask, 0], X_moons[mask, 1], c=[color], label=name, s=25, alpha=0.75)

ax2.set_title('DBSCAN vs K-Means: Non-Convex Shapes\n(Half-Moons — where DBSCAN shines)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Feature 1', fontsize=11)
ax2.set_ylabel('Feature 2', fontsize=11)
ax2.legend(framealpha=0.9)

plt.tight_layout()
plt.savefig('dbscan_clusters_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as dbscan_clusters_visualization.png')

## Part 4: Hyperparameter Experiments
DBSCAN has two critical hyperparameters: **ε** (neighbourhood radius) and **MinPts** (minimum samples). The k-distance plot is the standard heuristic for choosing ε. MinPts is typically set to 2×dimensions or higher for noisy data.

In [ ]:
# ── CELL 13: HYPERPARAMETER EXPERIMENTS ──────────────────────────────────────

from sklearn.neighbors import NearestNeighbors

# ── Experiment 1: k-distance plot to choose ε ──
# The 'elbow' in this plot is the optimal ε value
k = 5  # Typically set to MinPts
nbrs = NearestNeighbors(n_neighbors=k).fit(X)
distances, _ = nbrs.kneighbors(X)
k_distances = np.sort(distances[:, k-1])[::-1]  # Sort descending

# ── Experiment 2: ε sweep — track clusters and noise ──
eps_values = np.arange(0.1, 1.2, 0.1)
results = []
for eps_val in eps_values:
    labels_exp = DBSCAN(eps=eps_val, min_samples=5).fit_predict(X)
    n_cl = len(set(labels_exp)) - (1 if -1 in labels_exp else 0)
    n_no = int(np.sum(labels_exp == -1))
    results.append({'eps': round(eps_val, 2), 'n_clusters': n_cl, 'n_noise': n_no})

results_df = pd.DataFrame(results)

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# k-distance plot
ax1 = axes[0]
ax1.plot(range(len(k_distances)), k_distances, color='steelblue', linewidth=2)
ax1.axhline(y=0.4, color='red', linestyle='--', alpha=0.7, label='ε = 0.4 (chosen)')
ax1.set_title(f'k-Distance Graph (k={k}) — Find the Elbow for ε', fontsize=13, fontweight='bold')
ax1.set_xlabel('Points sorted by distance (descending)', fontsize=11)
ax1.set_ylabel(f'{k}-th Nearest Neighbour Distance', fontsize=11)
ax1.legend()

# ε sweep: clusters and noise
ax2 = axes[1]
color1, color2 = 'steelblue', 'tomato'
l1, = ax2.plot(results_df['eps'], results_df['n_clusters'], 'o-', color=color1, label='Num Clusters', linewidth=2)
ax2b = ax2.twinx()
l2, = ax2b.plot(results_df['eps'], results_df['n_noise'], 's--', color=color2, label='Noise Points', linewidth=2)
ax2.set_title('Effect of ε on Cluster Count and Noise\n(MinPts=5, full dataset)', fontsize=13, fontweight='bold')
ax2.set_xlabel('ε (eps)', fontsize=11)
ax2.set_ylabel('Number of Clusters', color=color1, fontsize=11)
ax2b.set_ylabel('Noise Points', color=color2, fontsize=11)
ax2.legend(handles=[l1, l2], loc='upper right')

plt.tight_layout()
plt.savefig('dbscan_hyperparameter_experiments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hyperparameter figure saved.')
print('\nε sweep summary:')
print(results_df.to_string(index=False))

In [ ]:
# ── CELL 14: CROSS-VALIDATION (MinPts sweep with silhouette) ─────────────────

# Vary MinPts, fix ε at optimal value found from k-distance plot
minpts_values = [3, 4, 5, 7, 10, 15]
sil_scores = []

for mp in minpts_values:
    lbl = DBSCAN(eps=0.4, min_samples=mp).fit_predict(X)
    n_cl = len(set(lbl)) - (1 if -1 in lbl else 0)
    m = lbl != -1
    if n_cl > 1 and m.sum() > 1:
        sil = silhouette_score(X[m], lbl[m])
    else:
        sil = np.nan
    sil_scores.append({'min_samples': mp, 'n_clusters': n_cl, 'silhouette': round(sil, 4) if not np.isnan(sil) else 'N/A'})

print('MinPts Sweep (eps=0.4):')
print(pd.DataFrame(sil_scores).to_string(index=False))

## Part 5: Interview Corner

### Q: Why does DBSCAN fail in high dimensions?
In high dimensions, the **curse of dimensionality** causes all pairwise distances to converge — the ratio of max to min distance approaches 1. This makes ε-neighbourhood practically meaningless because all points become equidistant. HDBSCAN partially mitigates this via mutual reachability distance, but the fundamental problem remains above ~10 dimensions. For high-D data, dimensionality reduction (PCA, UMAP) before clustering is standard practice.

### Q: How do you choose ε and MinPts?
- **MinPts**: Rule of thumb = max(2×dimensions, 5). Higher MinPts for noisy data.
- **ε**: Use the **k-distance plot** (k = MinPts). Sort the k-th nearest neighbour distances and find the 'elbow' — this is your ε. The elbow represents the point where distances increase sharply, separating dense from sparse regions.

### Q: DBSCAN vs K-Means — when to use which?
| Property | DBSCAN | K-Means |
|---|---|---|
| Cluster shape | Arbitrary | Convex only |
| Specify k? | No | Yes |
| Handles noise? | Yes (−1) | No |
| Scale sensitive? | Yes | Yes |
| Guarantees | None | Local optimum |

### Q: DBSCAN vs HDBSCAN?
DBSCAN requires a **global** ε — if clusters have varying densities, a single ε cannot capture all of them. HDBSCAN constructs a **hierarchy of clusterings** over all possible ε values and extracts the most stable clusters automatically. It is strictly more powerful but slower and harder to interpret.

## Key Takeaways — 5 Bullets for Placement Interviews

1. **No k required:** DBSCAN discovers the number of clusters automatically, unlike K-Means or GMMs — a critical advantage when k is unknown.

2. **Noise handling is built-in:** Points labelled −1 are genuine outliers, making DBSCAN a natural anomaly detector in addition to a clustering algorithm.

3. **Scale everything first:** DBSCAN uses Euclidean distance; unscaled features cause dominant-scale features to absorb the ε-neighbourhood, breaking the algorithm entirely.

4. **HDBSCAN is the production-grade upgrade:** It handles varying-density clusters, is more robust to MinPts choice, and provides soft (probabilistic) cluster membership — always prefer HDBSCAN in practice.

5. **Curse of dimensionality limits applicability:** DBSCAN degrades significantly beyond ~10 features; always pair with UMAP or PCA for high-dimensional data (text embeddings, image features).